# Embeddings treinadas do zero

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from collections import Counter
import math

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Corpus formado por caracteres


In [3]:

device = "cuda" if torch.cuda.is_available() else "cpu"
SEQ_LENGHT = 25       # tamanho da sequencia de entrada (quantos caracteres o modelo "ve" por vez)
HIDDEN_SIZE = 128     # tamanho do estado oculto da LSTM (memoria)
EMBEDDING_SIZE = 64   # dimensão dos vetores de embedding
LEARNING_RATE = 0.001 #
EPOCHS = 25
BATCH_SIZE = 64



def load_corpus(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read().lower()
    return text


def create_vocabulary(text, min_freq=2):
    """
    cria o vocabulario a partir do texto.
    inclui apenas caracteres que aparecem com frequência >= min_freq.
    adiciona tokens especiais para lidar com casos especificos.
    """
    char_counts = Counter(text)
    vocab = [char for char, count in char_counts.items() if count >= min_freq]

    special_tokens = ["<pad>", "<sos>", "<eos>", "<unk>"]
    vocab = special_tokens + vocab

    # dicionarios para conversao
    char_to_idx = {char: idx for idx, char in enumerate(vocab)}
    idx_to_char = {idx: char for idx, char in enumerate(vocab)}

    return vocab, char_to_idx, idx_to_char


def text_to_tensor(text, char_to_idx, seq_length=SEQ_LENGHT):
    """
    converte o texto em pares (input_seq, target_seq).
    cada input_seq tem tamanho seq_length.
    o target_seq é a mesma sequência deslocada em 1 caractere.
    exemplo:
        texto: "amor"
        input: "amo"  -> target: "mor"
    """
    input_seqs, target_seqs = [], []
    for i in range(0, len(text) - seq_length):
        input_seq = text[i:i+seq_length]
        target_seq = text[i+1:i+seq_length+1]

        seq_ids = [char_to_idx.get(c, char_to_idx["<unk>"]) for c in input_seq]
        target_ids = [char_to_idx.get(c, char_to_idx["<unk>"]) for c in target_seq]

        input_seqs.append(torch.tensor(seq_ids, dtype=torch.long))
        target_seqs.append(torch.tensor(target_ids, dtype=torch.long))
    return input_seqs, target_seqs


class SimpleLSTM(nn.Module):
    """
    modelo de linguagem baseado em LSTM:
    - embedding: converte indices em vetores densos.
    - LSTM: processa sequencias de embeddings e mantém contexto temporal.
    - linear: projeta saída da LSTM no espaço do vocabulario.
    """
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.lstm = nn.LSTM(embedding_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)              # [batch_size, seq_len, embedding_size]
        lstm_out, hidden = self.lstm(embedded, hidden)  # [batch_size, seq_len, hidden_size]
        output = self.fc(lstm_out)                # [batch_size, seq_len, vocab_size]
        return output, hidden



def generate_text(model, idx_to_char, char_to_idx, start_text, length=100, temperature=1.0):
    """
    gera texto a partir de um prompt inicial.
    - start_text: texto de entrada usado como "semente".
    - length: tamanho do texto a ser gerado.
    - temperature: controla a aleatoriedade.
    """
    model.eval()
    chars = [c for c in start_text.lower()]
    input_seq = torch.tensor([char_to_idx.get(c, char_to_idx['<unk>']) for c in chars]).unsqueeze(0).to(device)
    generated = chars.copy()
    hidden = None

    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(input_seq, hidden)
            last_logits = output[0, -1, :] / temperature
            probabilities = F.softmax(last_logits, dim=-1)

            # amostra o proximo caractere proporcional as probabilidades
            next_char_idx = torch.multinomial(probabilities, 1).item()
            next_char = idx_to_char[next_char_idx]

            generated.append(next_char)
            input_seq = torch.tensor([[next_char_idx]]).to(device)

    return ''.join(generated)


def generate_with_temperatures(model, idx_to_char, char_to_idx, prompts, temperatures=[0.7, 1.0, 1.3], length=200):
    """
    gera textos com diferentes temparaturas e com diferentes prompts
    """

    for prompt in prompts:
        print(f"Prompt: '{prompt}'")
        for temp in temperatures:
            generated = generate_text(model, idx_to_char, char_to_idx, prompt, length=length, temperature=temp)
            print(f"[Temperature = {temp}]")
            print(generated)
            print("-" * 80)
        print("=" * 100)

def evaluate_model(model, data, criterion):
    """
    avalia o modelo em um conjunto de dados.
    Retorna:
    - perda media (CrossEntropyLoss)
    - perplexidade (exp da perda media)
    """
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for i in range(0, len(data[0]), BATCH_SIZE):
            input_batch = torch.stack(data[0][i:i+BATCH_SIZE]).to(device)
            target_batch = torch.stack(data[1][i:i+BATCH_SIZE]).to(device)

            output, _ = model(input_batch)
            loss = criterion(output.view(-1, model.vocab_size), target_batch.view(-1))
            total_loss += loss.item()

    avg_loss = total_loss / (len(data[0]) / BATCH_SIZE)
    ppl = math.exp(avg_loss)  # perplexidade é o expoente da perda
    return avg_loss, ppl



def train_model(model, train_data, val_data, epochs=EPOCHS):
    """
    treina o modelo no conjunto de treino.
    a cada epoca calcula:
    - loss medio no treino e perplexidade.
    - Loss medio no validacao e perplexidade.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        # loop nos batches do treino
        for i in range(0, len(train_data[0]), BATCH_SIZE):
            input_batch = torch.stack(train_data[0][i:i+BATCH_SIZE]).to(device)
            target_batch = torch.stack(train_data[1][i:i+BATCH_SIZE]).to(device)

            optimizer.zero_grad()
            output, _ = model(input_batch)
            loss = criterion(output.view(-1, model.vocab_size), target_batch.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # metricas de treino
        avg_train_loss = total_loss / (len(train_data[0]) / BATCH_SIZE)
        train_ppl = math.exp(avg_train_loss)

        # metricas de validação
        val_loss, val_ppl = evaluate_model(model, val_data, criterion)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {avg_train_loss:.4f}, Train PPL: {train_ppl:.2f} | "
              f"Val Loss: {val_loss:.4f}, Val PPL: {val_ppl:.2f}")



print("uploading corpus...")
text = load_corpus("/content/corpus.txt")

print("creating vocabulary...")
vocab, char_to_idx, idx_to_char = create_vocabulary(text)

print("preparing sequences...")
sequences, targets = text_to_tensor(text, char_to_idx)

# divisão treino/val/teste
total_size = len(sequences)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
train_data = (sequences[:train_size], targets[:train_size])
val_data = (sequences[train_size:train_size+val_size], targets[train_size:train_size+val_size])
test_data = (sequences[train_size+val_size:], targets[train_size+val_size:])

print(f"Train: {len(train_data[0])}, Val: {len(val_data[0])}, Test: {len(test_data[0])}")

# cria modelo
print("creating model...")
model = SimpleLSTM(len(vocab), EMBEDDING_SIZE, HIDDEN_SIZE).to(device)

# treina modelo
print("training...")
train_model(model, train_data, val_data)

# avaliação no teste final
criterion = nn.CrossEntropyLoss()
test_loss, test_ppl = evaluate_model(model, test_data, criterion)
print(f"\nFinal Test Loss: {test_loss:.4f}, Test PPL: {test_ppl:.2f}")

# gera alguns textos com 3 tipos de temperatura
prompts = ["o amor e", "a vida e", "eu sinto"]
generate_with_temperatures(model, idx_to_char, char_to_idx, prompts, temperatures=[0.7, 1.0, 1.3], length=200)


uploading corpus...
creating vocabulary...
preparing sequences...
Train: 295650, Val: 36956, Test: 36957
creating model...
training...
Epoch 1/25 | Train Loss: 1.9821, Train PPL: 7.26 | Val Loss: 1.7881, Val PPL: 5.98
Epoch 2/25 | Train Loss: 1.7861, Train PPL: 5.97 | Val Loss: 1.6972, Val PPL: 5.46
Epoch 3/25 | Train Loss: 1.7098, Train PPL: 5.53 | Val Loss: 1.6521, Val PPL: 5.22
Epoch 4/25 | Train Loss: 1.6617, Train PPL: 5.27 | Val Loss: 1.6277, Val PPL: 5.09
Epoch 5/25 | Train Loss: 1.6265, Train PPL: 5.09 | Val Loss: 1.6150, Val PPL: 5.03
Epoch 6/25 | Train Loss: 1.5996, Train PPL: 4.95 | Val Loss: 1.6082, Val PPL: 4.99
Epoch 7/25 | Train Loss: 1.5787, Train PPL: 4.85 | Val Loss: 1.6035, Val PPL: 4.97
Epoch 8/25 | Train Loss: 1.5613, Train PPL: 4.76 | Val Loss: 1.6000, Val PPL: 4.95
Epoch 9/25 | Train Loss: 1.5466, Train PPL: 4.70 | Val Loss: 1.5998, Val PPL: 4.95
Epoch 10/25 | Train Loss: 1.5340, Train PPL: 4.64 | Val Loss: 1.6009, Val PPL: 4.96
Epoch 11/25 | Train Loss: 1.5232, 

## Corpus formado por palavras

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
import math
import re
import unicodedata
from torch.utils.data import DataLoader, TensorDataset


device = "cuda" if torch.cuda.is_available() else "cpu"

SEQ_LENGTH = 30  # comprimento das sequencias de entrada
HIDDEN_SIZE = 128  # Tamanho da camada oculta da LSTM
EMBEDDING_SIZE = 64  # Dimensão dos embeddings de palavras
LEARNING_RATE = 0.001
EPOCHS = 25
BATCH_SIZE = 128  #
MIN_FREQ = 3  # frequencia que a palavra deve ter para entrar no vocabulario

# realize uma normalzacao no textp
def clean_text(text):

    text = unicodedata.normalize('NFKD', text.lower())
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def load_corpus(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    return clean_text(text)

# cria todo o vocabulario formado por palavras
def create_vocabulary(text, min_freq=MIN_FREQ):
    # divide o texto em palavras individuais
    words = text.split()
    # conta a freq de cada palavra
    word_counts = Counter(words)

    # aplica um filtro nas palavras pelo freqeuncia minima = 3
    vocab_words = [word for word, count in word_counts.items() if count >= min_freq]

    # token especiais
    special_tokens = ["<pad>", "<unk>", "<sos>", "<eos>"]
    vocab = special_tokens + sorted(vocab_words)

    # cria o mapeamento palavra -> indice
    word_to_idx = {word: idx for idx, word in enumerate(vocab)}
    # cria mapeamento indice -> palavra
    idx_to_word = {idx: word for word, idx in word_to_idx.items()}

    # retorna o vocabulrario criado, bem como os dicionarios de palavra <--> indice
    return vocab, word_to_idx, idx_to_word

# prepara as sequencias de treinamento
def prepare_sequences(text, word_to_idx, seq_length=SEQ_LENGTH):
    words = text.split()
    input_seqs = []
    target_seqs = []

    # percorre o texto criando sequrncias sobrepostas
    for i in range(len(words) - seq_length):

        # pega sequencia de palavras do vocabulario
        input_words = words[i:i + seq_length]
        target_words = words[i + 1:i + seq_length + 1]

        # converte pra indice e vice versa
        input_ids = [word_to_idx.get(w, word_to_idx["<unk>"]) for w in input_words]
        target_ids = [word_to_idx.get(w, word_to_idx["<unk>"]) for w in target_words]

        # adiciona as sequencias as listas
        input_seqs.append(input_ids)
        target_seqs.append(target_ids)

    return input_seqs, target_seqs

class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()

        # camada de embedding: converte indices em vetores densos
        self.embedding = nn.Embedding(vocab_size, embedding_size)

        # camada LSTM: processa sequencias de embeddings
        self.lstm = nn.LSTM(embedding_size, hidden_size, batch_first=True)

        # camada linear final: projeta para o tamanho do vocabulário
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        # converte indices em embeddings
        x = self.embedding(x)

        # proocessa atraves da camada LSTM
        lstm_out, hidden = self.lstm(x, hidden)

        output = self.fc(lstm_out)
        return output, hidden

# calcula a medida de perplexidade do modelo
def calculate_ppl(model, data_loader):

    model.eval()

    # define a função de perda (entropia cruzada)
    criterion = nn.CrossEntropyLoss()
    total_loss = 0
    total_tokens = 0

    with torch.no_grad():

        # percorre sobre todos os lotes do data loader
        for inputs, targets in data_loader:
            # move dados para o dispositivo (GPU/CPU)
            inputs, targets = inputs.to(device), targets.to(device)

            # forward pass - obtem as previsoes que o modelo gera
            outputs, _ = model(inputs)

            # calcula a loss (entropia cruzada)
            loss = criterion(outputs.view(-1, outputs.size(-1)), targets.view(-1))

            # acumula a loss ponderada pelo numero de tokens
            batch_tokens = targets.numel()  # numero total de tokens do lote
            total_loss += loss.item() * batch_tokens
            total_tokens += batch_tokens

    # caalcula a loss media por token
    avg_loss = total_loss / total_tokens

    # calcula perplexidade: exponencial da loss media
    ppl = math.exp(avg_loss)

    return ppl, avg_loss

# funcao de treinamento do modelo
def train_model(model, train_loader, val_loader, epochs=EPOCHS):

    # deefine  adam como otimizador
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    # define a funcao de perda
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float('inf')

    for epoch in range(epochs):
        # treino
        model.train()
        train_loss = 0
        for inputs, targets in train_loader:
            # move dados para dispositivo
            inputs, targets = inputs.to(device), targets.to(device)

            # zera gradientes da epoca anterior
            optimizer.zero_grad()
            # Forward pass
            outputs, _ = model(inputs)
            # Calcula loss
            loss = criterion(outputs.view(-1, outputs.size(-1)), targets.view(-1))
            # Backward pass (calcula gradientes)
            loss.backward()
            # Atualiza pesos
            optimizer.step()

            # Acumula loss
            train_loss += loss.item()

        # validacao
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs, _ = model(inputs)
                loss = criterion(outputs.view(-1, outputs.size(-1)), targets.view(-1))
                val_loss += loss.item()

        # calcula loss media
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)

        # callcula PPL para treino e validaçao
        train_ppl, _ = calculate_ppl(model, train_loader)
        val_ppl, _ = calculate_ppl(model, val_loader)


        print(f"Época {epoch+1}/{epochs}:")
        print(f"  Train Loss: {train_loss:.4f} | Train PPL: {train_ppl:.2f}")
        print(f"  Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f}")

        # salva o melhor modelo ate o momento
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            print("  Melhor modelo salvo!")

def generate_text(model, idx_to_word, word_to_idx, start_text, length=30, temperature=1.0):

    model.eval()
    # limpa e divide o texto inicial
    words = clean_text(start_text).split()

    # converte palavras iniciais para indices
    input_seq = [word_to_idx.get(w, word_to_idx['<unk>']) for w in words]
    # Converte para tensor e adiciona dimensão de batch
    input_seq = torch.tensor(input_seq).unsqueeze(0).to(device)

    # inicia a lista de palavras geradas
    generated = words.copy()
    hidden = None

    # lista de tokens a remover durante geração
    remove_tokens = [word_to_idx[t] for t in ["<pad>", "<sos>", "<eos>", "<unk>"] if t in word_to_idx]

    with torch.no_grad():
        for _ in range(length):  # gera o 'length' palavras
            # forward pass no modelo
            outputs, hidden = model(input_seq, hidden)

            # pega logits da ultima palavra e aplica temperatura
            last_logits = outputs[0, -1, :] / temperature

            # remove probabilidade de tokens especiais
            if remove_tokens:
                last_logits[remove_tokens] = float('-inf')

            # converte logits em probabilidades
            probs = F.softmax(last_logits, dim=-1)
            next_idx = torch.multinomial(probs, 1).item()

            # converte indice para palavra
            next_word = idx_to_word[next_idx]
            generated.append(next_word)

            input_seq = torch.tensor([[next_idx]]).to(device)

    return ' '.join(generated)


def generate_with_temperatures(model, idx_to_word, word_to_idx, prompts, temperatures=[0.7, 1.0, 1.3], length=30):
    print("\n=== GERAÇÃO DE TEXTO COM DIFERENTES TEMPERATURAS ===\n")
    for prompt in prompts:
        print(f"Prompt: '{prompt}'")
        for temp in temperatures:
            gen = generate_text(model, idx_to_word, word_to_idx, prompt, length, temp)
            print(f"[Temperature = {temp}]")
            print(gen)
            print("-" * 80)
        print("=" * 100)


print("Carregando corpus...")
text = load_corpus("/content/corpus.txt")
print(f"Texto tem {len(text.split())} palavras")

print("Criando vocabulario...")
vocab, word_to_idx, idx_to_word = create_vocabulary(text)

print("Preparando sequecias...")

input_seqs, target_seqs = prepare_sequences(text, word_to_idx)

# divisao dos dados
total_seqs = len(input_seqs)
train_size = int(0.8 * total_seqs)
val_size = int(0.1 * total_seqs)

# Converte listas para tensores PyTorch
train_inputs = torch.tensor(input_seqs[:train_size], dtype=torch.long)
train_targets = torch.tensor(target_seqs[:train_size], dtype=torch.long)
val_inputs = torch.tensor(input_seqs[train_size:train_size+val_size], dtype=torch.long)
val_targets = torch.tensor(target_seqs[train_size:train_size+val_size], dtype=torch.long)
test_inputs = torch.tensor(input_seqs[train_size+val_size:], dtype=torch.long)
test_targets = torch.tensor(target_seqs[train_size+val_size:], dtype=torch.long)

print(f"Sequencias: Train={len(train_inputs)}, Val={len(val_inputs)}, Test={len(test_inputs)}")

# cria datasets pytorch
train_dataset = TensorDataset(train_inputs, train_targets)
val_dataset = TensorDataset(val_inputs, val_targets)
test_dataset = TensorDataset(test_inputs, test_targets)

# cria data loaders para carregamento em lotes
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Criando modelo...")
model = SimpleLSTM(len(vocab), EMBEDDING_SIZE, HIDDEN_SIZE)

# move pra gpu para realizar as operacoes
model = model.to(device)
print(f"Modelo movido para {device}")

# treinamento
print("Iniciando treinamento...")
train_model(model, train_loader, val_loader)

# avalizacao fimal com PPl
print("\nAvaliação final...")
model.load_state_dict(torch.load('best_model.pth'))

# calcula o PPL para todos os conjuntos montados
train_ppl, train_loss = calculate_ppl(model, train_loader)
val_ppl, val_loss = calculate_ppl(model, val_loader)
test_ppl, test_loss = calculate_ppl(model, test_loader)

print("\n=== RESULTADOS FINAIS ===")
print(f"Train - Loss: {train_loss:.4f} | PPL: {train_ppl:.2f}")
print(f"Val   - Loss: {val_loss:.4f} | PPL: {val_ppl:.2f}")
print(f"Test  - Loss: {test_loss:.4f} | PPL: {test_ppl:.2f}")

print("\nGerando textos...")
prompts = ["o amor e", "a vida e", "eu sinto"]
generate_with_temperatures(model, idx_to_word, word_to_idx, prompts)

# for prompt in prompts:
#     for temperature in [0.7, 1.0, 1.3]:
#         # gera texto continuando a partir de cada prompt
#         generated = generate_text(model, idx_to_word, word_to_idx, prompt, length=30, temperature=temperature)
#         print(f"Prompt: '{prompt}'")
#         print(f"Gerado: {generated}")

Carregando corpus...
Texto tem 72983 palavras
Criando vocabulario...
Preparando sequecias...
Sequencias: Train=58362, Val=7295, Test=7296
Criando modelo...
Modelo movido para cuda
Iniciando treinamento...
Época 1/25:
  Train Loss: 5.6073 | Train PPL: 158.10
  Val Loss: 5.7694 | Val PPL: 320.37
  Melhor modelo salvo!
Época 2/25:
  Train Loss: 4.7395 | Train PPL: 85.38
  Val Loss: 5.7175 | Val PPL: 304.15
  Melhor modelo salvo!
Época 3/25:
  Train Loss: 4.2379 | Train PPL: 56.22
  Val Loss: 5.8267 | Val PPL: 339.26
Época 4/25:
  Train Loss: 3.8679 | Train PPL: 40.27
  Val Loss: 5.9797 | Val PPL: 395.36
Época 5/25:
  Train Loss: 3.5577 | Train PPL: 30.16
  Val Loss: 6.1419 | Val PPL: 465.00
Época 6/25:
  Train Loss: 3.2863 | Train PPL: 23.38
  Val Loss: 6.3166 | Val PPL: 553.72
Época 7/25:
  Train Loss: 3.0464 | Train PPL: 18.64
  Val Loss: 6.4978 | Val PPL: 663.78
Época 8/25:
  Train Loss: 2.8327 | Train PPL: 15.24
  Val Loss: 6.6856 | Val PPL: 800.88
Época 9/25:
  Train Loss: 2.6425 | T

# Embeddings pré-treinadas


In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.pt.300.vec.gz

--2025-10-12 13:01:02--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.pt.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 99.84.41.33, 99.84.41.79, 99.84.41.129, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|99.84.41.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1271093660 (1.2G) [binary/octet-stream]
Saving to: ‘cc.pt.300.vec.gz’

cc.pt.300.vec.gz    100%[===================>]   1.18G  92.4MB/s    in 8.6s    

2025-10-12 13:01:10 (141 MB/s) - ‘cc.pt.300.vec.gz’ saved [1271093660/1271093660]



In [ ]:
!gunzip cc.pt.300.vec.gz

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from collections import Counter
import numpy as np
import random
import os
import re

device = "cuda" if torch.cuda.is_available() else "cpu"
SEQ_LENGTH = 30
HIDDEN_SIZE = 128            # menor tamanho -> menos overfitting
EMBEDDING_SIZE = 300
LEARNING_RATE = 0.0001
EPOCHS = 25
BATCH_SIZE = 128
FASTTEXT_VEC_PATH = "/content/cc.pt.300.vec"


def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def load_corpus(file_path):
    """Carrega o corpus de texto do arquivo"""
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    return clean_text(text).split()

def create_vocabulary(words, min_freq=3):
    # Cria o vocabulario e os mapeamentos palavra <-> indice.
    word_counts = Counter(words)
    vocab_list = [w for w, c in word_counts.items() if c >= min_freq]
    special_tokens = ["<pad>", "<sos>", "<eos>", "<unk>"]
    vocab = special_tokens + vocab_list
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    idx_to_word = {i: w for i, w in enumerate(vocab)}
    return vocab, word_to_idx, idx_to_word

def words_to_tensor(words, word_to_idx, seq_length=SEQ_LENGTH):
    """
    Transforma o corpus em pares (entrada, alvo) de sequência de palavras.
    Adiciona um pequeno deslocamento aleatório nas janelas para aumentar a diversidade.
    """
    inputs, targets = [], []
    i = 0
    while i < len(words) - seq_length:
        input_seq = words[i:i+seq_length]
        target_seq = words[i+1:i+seq_length+1]
        seq_ids = [word_to_idx.get(w, word_to_idx["<unk>"]) for w in input_seq]
        target_ids = [word_to_idx.get(w, word_to_idx["<unk>"]) for w in target_seq]
        inputs.append(torch.tensor(seq_ids, dtype=torch.long))
        targets.append(torch.tensor(target_ids, dtype=torch.long))
        i += random.randint(1, 3)  # deslocamento aleatório (data augmentation)
    return inputs, targets


# 2. Embeddings fastText

def load_fasttext_embeddings(vocab, word_to_idx, path_to_vec):
    """Carrega apenas os vetores fastText do vocabulário necessário."""
    print("Carregando embeddings fastText (somente palavras do vocabulário)...")
    vocab_set = set(vocab)
    vocab_size = len(vocab)
    embedding_matrix = np.random.normal(scale=0.6, size=(vocab_size, EMBEDDING_SIZE)).astype(np.float32)
    found = 0

    if not os.path.exists(path_to_vec):
        raise FileNotFoundError(f"Arquivo fastText não encontrado: {path_to_vec}")

    with open(path_to_vec, 'r', encoding='utf-8', newline='\n', errors='ignore') as f:
        first_line = f.readline().split()
        if len(first_line) != 2 or not all(x.isdigit() for x in first_line):
            f.seek(0)
        for line in f:
            parts = line.rstrip().split(' ')
            if len(parts) <= EMBEDDING_SIZE: continue
            word = parts[0]
            if word in vocab_set:
                vec = np.asarray(parts[1:], dtype='float32')
                if vec.shape[0] == EMBEDDING_SIZE:
                    idx = word_to_idx[word]
                    embedding_matrix[idx] = vec
                    found += 1
    print(f"Vetores encontrados: {found}/{len(vocab)} ({found/len(vocab)*100:.1f}%)")
    return torch.tensor(embedding_matrix, dtype=torch.float32)

# 3. Modelo LSTM com dropout
class LSTM_FastText(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pretrained_weights, freeze_embeddings=True):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        # Embedding pré-treinado
        self.embedding = nn.Embedding.from_pretrained(pretrained_weights, freeze=freeze_embeddings)
        # Dropout na LSTM ajuda a reduzir overfitting
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        output, hidden = self.lstm(x, hidden)
        output = self.fc(output)
        return output, hidden

# 4. Treinamento e avaliação
def evaluate_model(model, data, criterion):
    """Avalia o modelo e retorna perda e perplexidade."""
    model.eval()
    total_loss, num_batches = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(data[0]), BATCH_SIZE):
            xb = torch.stack(data[0][i:i+BATCH_SIZE]).to(device)
            yb = torch.stack(data[1][i:i+BATCH_SIZE]).to(device)
            output, _ = model(xb)
            loss = criterion(output.view(-1, model.vocab_size), yb.view(-1))
            total_loss += loss.item()
            num_batches += 1
    avg_loss = total_loss / max(1, num_batches)
    return avg_loss, math.exp(avg_loss)

def train_model(model, train_data, val_data, epochs=EPOCHS, unfreeze_after=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs+1):
        model.train()
        total_loss, num_batches = 0.0, 0
        for i in range(0, len(train_data[0]), BATCH_SIZE):
            xb = torch.stack(train_data[0][i:i+BATCH_SIZE]).to(device)
            yb = torch.stack(train_data[1][i:i+BATCH_SIZE]).to(device)
            optimizer.zero_grad()
            output, _ = model(xb)
            loss = criterion(output.view(-1, model.vocab_size), yb.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item()
            num_batches += 1

        avg_train_loss = total_loss / max(1, num_batches)
        train_ppl = math.exp(avg_train_loss)
        val_loss, val_ppl = evaluate_model(model, val_data, criterion)

        print(f"Epoch {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f}, PPL: {train_ppl:.2f} | "
              f"Val Loss: {val_loss:.4f}, PPL: {val_ppl:.2f}")

        if epoch == unfreeze_after:
            model.embedding.requires_grad_(True)
            print(f"Embeddings descongelados para fine-tuning na época {epoch}.")

# 5. Geração de texto
def generate_text(model, idx_to_word, word_to_idx, start_text, length=30, temperature=1.0):
    """Gera texto a partir de um prompt, ajustando a diversidade pela temperatura."""
    model.eval()
    words = start_text.lower().split() or ["<sos>"]
    input_seq = torch.tensor([[word_to_idx.get(w, word_to_idx["<unk>"]) for w in words]], dtype=torch.long).to(device)
    generated = words.copy()
    hidden = None

    remove_tokens = [word_to_idx[t] for t in ["<pad>", "<sos>", "<eos>", "<unk>"] if t in word_to_idx]

    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(input_seq, hidden)
            logits = output[0, -1, :] / temperature

            if remove_tokens: # Bloqueia tokens especiais
                logits[remove_tokens] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, 1).item()
            next_word = idx_to_word[next_idx]
            generated.append(next_word)
            input_seq = torch.tensor([[next_idx]], dtype=torch.long).to(device)
    return ' '.join(generated)

def generate_with_temperatures(model, idx_to_word, word_to_idx, prompts, temperatures=[0.7, 1.0, 1.3], length=30):
    print("\n=== GERAÇÃO DE TEXTO COM DIFERENTES TEMPERATURAS ===\n")
    for prompt in prompts:
        print(f"Prompt: '{prompt}'")
        for temp in temperatures:
            gen = generate_text(model, idx_to_word, word_to_idx, prompt, length, temp)
            print(f"[Temperature = {temp}]")
            print(gen)
            print("-" * 80)
        print("=" * 100)


print("Carregando corpus...")
words = load_corpus("/content/corpus.txt")

print("Criando vocabulário...")
vocab, word_to_idx, idx_to_word = create_vocabulary(words)

print("Preparando sequências...")
inputs, targets = words_to_tensor(words, word_to_idx)
total = len(inputs)
train_size = int(0.8 * total)
val_size = int(0.1 * total)
train_data = (inputs[:train_size], targets[:train_size])
val_data = (inputs[train_size:train_size+val_size], targets[train_size:train_size+val_size])
test_data = (inputs[train_size+val_size:], targets[train_size+val_size:])
print(f"Train: {len(train_data[0])}, Val: {len(val_data[0])}, Test: {len(test_data[0])}")

print("Carregando embeddings fastText...")
embedding_matrix = load_fasttext_embeddings(vocab, word_to_idx, FASTTEXT_VEC_PATH)

print("Criando modelo...")
model = LSTM_FastText(len(vocab), EMBEDDING_SIZE, HIDDEN_SIZE, embedding_matrix, freeze_embeddings=True).to(device)

print(f"Treinando modelo ({EPOCHS} épocas)...")
train_model(model, train_data, val_data, epochs=EPOCHS, unfreeze_after=5)

print("Avaliando no conjunto de teste...")
criterion = nn.CrossEntropyLoss()
test_loss, test_ppl = evaluate_model(model, test_data, criterion)
print(f"Final Test Loss: {test_loss:.4f}, Test PPL: {test_ppl:.2f}")

prompts = ["o amor e", "a vida e", "eu sinto"]
generate_with_temperatures(model, idx_to_word, word_to_idx, prompts)


Carregando corpus...
Criando vocabulário...
Preparando sequências...
Train: 29124, Val: 3640, Test: 3642
Carregando embeddings fastText...
Carregando embeddings fastText (somente palavras do vocabulário)...
Vetores encontrados: 2703/2730 (99.0%)
Criando modelo...
Treinando modelo (25 épocas)...
Epoch 1/25 | Train Loss: 6.9155, PPL: 1007.75 | Val Loss: 6.2461, PPL: 516.00
Epoch 2/25 | Train Loss: 5.9478, PPL: 382.92 | Val Loss: 6.1396, PPL: 463.89
Epoch 3/25 | Train Loss: 5.8858, PPL: 359.89 | Val Loss: 6.1250, PPL: 457.16
Epoch 4/25 | Train Loss: 5.8624, PPL: 351.56 | Val Loss: 6.1213, PPL: 455.44
Epoch 5/25 | Train Loss: 5.8475, PPL: 346.35 | Val Loss: 6.1215, PPL: 455.55
Embeddings descongelados para fine-tuning na época 5.
Epoch 6/25 | Train Loss: 5.8344, PPL: 341.86 | Val Loss: 6.1200, PPL: 454.85
Epoch 7/25 | Train Loss: 5.8234, PPL: 338.13 | Val Loss: 6.1200, PPL: 454.88
Epoch 8/25 | Train Loss: 5.8155, PPL: 335.46 | Val Loss: 6.1210, PPL: 455.34
Epoch 9/25 | Train Loss: 5.8093, 